# BERT Text Embedding Extraction (IEMOCAP)


## 1) Setup, Paths, and Run Configuration


In [1]:
from __future__ import annotations

import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('Could not locate repository root (missing pyproject.toml).')


REPO_ROOT = find_repo_root(Path.cwd())
META_CSV = REPO_ROOT / 'datasets' / 'IEMOCAP' / 'iemocap_full_dataset.csv'
IEMOCAP_ROOT = REPO_ROOT / 'datasets' / 'IEMOCAP'
OUT_DIR = REPO_ROOT / 'extracted_features' / 'text'
EMB_CSV = OUT_DIR / 'bert_embeddings.csv'
META_JSON = OUT_DIR / 'bert_embeddings_metadata.json'

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
MODEL_NAME = 'bert-base-uncased'
MAX_LENGTH = 96
BATCH_SIZE = 64
POOLING = 'mean'  # 'mean' or 'cls'
INCLUDE_XXX = True
REQUIRE_AGREEMENT = True
EXCLUDED_EMOTIONS = {'sur', 'fea', 'oth', 'dis'}
USE_FP16_IF_CUDA = True
MAX_ROWS = None  # set int (for quick experiments), e.g. 1000
TRANSCRIPT_WORKERS = min(8, max(1, os.cpu_count() or 1))

print(f'Repo root: {REPO_ROOT}')
print(f'Metadata CSV: {META_CSV}')
print(f'Output embedding CSV: {EMB_CSV}')
print(f'Output metadata JSON: {META_JSON}')
print('Run config:')
print({
    'model': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'pooling': POOLING,
    'include_xxx': INCLUDE_XXX,
    'require_agreement': REQUIRE_AGREEMENT,
    'max_rows': MAX_ROWS,
    'transcript_workers': TRANSCRIPT_WORKERS,
})


Repo root: F:\Speech-Emotion-Recognition
Metadata CSV: F:\Speech-Emotion-Recognition\datasets\IEMOCAP\iemocap_full_dataset.csv
Output embedding CSV: F:\Speech-Emotion-Recognition\extracted_features\text\bert_embeddings.csv
Output metadata JSON: F:\Speech-Emotion-Recognition\extracted_features\text\bert_embeddings_metadata.json
Run config:
{'model': 'bert-base-uncased', 'max_length': 96, 'batch_size': 64, 'pooling': 'mean', 'include_xxx': True, 'require_agreement': True, 'max_rows': None, 'transcript_workers': 8}


## 2) Build Utterance-to-Text Index from Transcripts


In [2]:
LINE_RE = re.compile(r'^(?P<utt>\S+)\s+\[[^\]]+\]:\s*(?P<text>.*)$')


def parse_transcript_file(txt_path: str) -> dict[str, str]:
    file_index: dict[str, str] = {}
    with Path(txt_path).open('r', encoding='utf-8', errors='ignore') as handle:
        for line in handle:
            m = LINE_RE.match(line.strip())
            if not m:
                continue
            utt = m.group('utt')
            text = m.group('text').strip()
            if not text:
                continue
            if utt in file_index:
                file_index[utt] = (file_index[utt] + ' ' + text).strip()
            else:
                file_index[utt] = text
    return file_index


def build_transcript_index(iemocap_dir: Path, max_workers: int = 4) -> dict[str, str]:
    files = sorted(iemocap_dir.glob('Session*/dialog/transcriptions/*.txt'))
    if not files:
        raise FileNotFoundError('No transcript files found under Session*/dialog/transcriptions/*.txt')

    workers = max(1, min(max_workers, len(files)))
    print(f'Parsing {len(files)} transcript files with {workers} thread workers...')
    start = time.perf_counter()

    partial_indexes: list[dict[str, str]] = []
    if workers == 1:
        for p in files:
            partial_indexes.append(parse_transcript_file(str(p)))
    else:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            for partial in ex.map(parse_transcript_file, [str(p) for p in files]):
                partial_indexes.append(partial)

    index: dict[str, str] = {}
    for partial in partial_indexes:
        for utt, text in partial.items():
            if utt in index:
                index[utt] = (index[utt] + ' ' + text).strip()
            else:
                index[utt] = text

    elapsed = time.perf_counter() - start
    print(f'Transcript indexing complete: {len(index):,} utterances ({elapsed:.2f}s)')
    return index


## 3) Load Metadata and Attach Text


In [3]:
df = pd.read_csv(META_CSV)
print(f'Loaded metadata rows: {len(df):,}')

# Normalize
df['emotion'] = df['emotion'].astype(str).str.strip().str.lower()
df['method'] = df['method'].astype(str).str.strip().str.lower()
df['gender'] = df['gender'].astype(str).str.strip().str.upper()

raw_rows = len(df)

# Exclude selected classes regardless of agreement.
df = df[~df['emotion'].isin(EXCLUDED_EMOTIONS)].copy()

# Include xxx by default. If INCLUDE_XXX is disabled, drop it explicitly.
if not INCLUDE_XXX:
    df = df[df['emotion'] != 'xxx'].copy()

# Keep agreement filter for labeled classes; preserve xxx rows.
if REQUIRE_AGREEMENT:
    df = df[(df['emotion'] == 'xxx') | (df['agreement'] > 0)].copy()

df['utt_id'] = df['path'].apply(lambda p: Path(p).stem)

print(f'Rows after filters: {len(df):,} ({len(df)/raw_rows:.2%} retained)')
print('Emotion distribution:')
print(df['emotion'].value_counts())

transcript_index = build_transcript_index(IEMOCAP_ROOT, max_workers=TRANSCRIPT_WORKERS)
df['text'] = df['utt_id'].map(transcript_index)

has_text = df['text'].notna() & (df['text'].str.len() > 0)
df_text = df[has_text].copy()

print(f'Rows with text: {len(df_text):,} / {len(df):,} ({has_text.mean():.2%})')
print('Session distribution:')
print(df_text['session'].value_counts().sort_index())
print('Method distribution:')
print(df_text['method'].value_counts())

if MAX_ROWS is not None:
    df_text = df_text.head(int(MAX_ROWS)).copy()
    print(f'Applied MAX_ROWS={MAX_ROWS}; using {len(df_text):,} rows')

df_text[['utt_id', 'emotion', 'text']].head()


Loaded metadata rows: 10,039
Rows after filters: 9,887 (98.49% retained)
Emotion distribution:
emotion
xxx    2507
fru    1849
neu    1708
ang    1103
sad    1084
exc    1041
hap     595
Name: count, dtype: int64
Parsing 151 transcript files with 8 thread workers...


Transcript indexing complete: 10,084 utterances (1.21s)
Rows with text: 9,887 / 9,887 (100.00%)
Session distribution:
session
1    1780
2    1784
3    2105
4    2076
5    2142
Name: count, dtype: int64
Method distribution:
method
script    5173
impro     4714
Name: count, dtype: int64


,utt_id,emotion,text
0,Ses01F_script02_1_F000,neu,Fine.
1,Ses01F_script02_1_F001,fru,[BREATHING]
2,Ses01F_script02_1_F002,xxx,What?
4,Ses01F_script02_1_F004,neu,That's not your flashlight.
5,Ses01F_script02_1_F005,xxx,It's ours; it's my flashlight too.


## 4) Train/Test Split Stats and Text Length Diagnostics


In [4]:
train_sessions = [1, 2, 3, 4]
test_sessions = [5]
df_text['split'] = df_text['session'].apply(lambda s: 'train' if s in train_sessions else 'test')

print('Split counts:')
print(df_text['split'].value_counts())
print('\nSplit x emotion crosstab:')
print(pd.crosstab(df_text['split'], df_text['emotion']))

char_len = df_text['text'].str.len()
print('\nCharacter-length summary:')
print(char_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


Split counts:
split
train    7745
test     2142
Name: count, dtype: int64

Split x emotion crosstab:
emotion  ang  exc   fru  hap   neu  sad   xxx
split                                        
test     170  299   381  143   384  245   520
train    933  742  1468  452  1324  839  1987

Character-length summary:
count    9887.000000
mean       59.036209
std        51.668915
min         1.000000
50%        43.000000
90%       130.000000
95%       162.000000
99%       227.000000
max       554.000000
Name: text, dtype: float64


## 5) Load Tokenizer and BERT Model


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

try:
    from transformers import AutoModel, AutoTokenizer
except Exception as exc:
    raise ImportError(
        'Failed to import transformers stack. '
        'Please update dependencies, e.g. `pip install -U transformers huggingface_hub` '
        f'and retry. Original error: {exc}'
    ) from exc

load_start = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
model.to(device)
load_elapsed = time.perf_counter() - load_start

hidden_size = int(model.config.hidden_size)
print(f'Model loaded: {MODEL_NAME}')
print(f'Hidden size: {hidden_size}')
print(f'Model/tokenizer load time: {load_elapsed:.2f}s')


Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: bert-base-uncased
Hidden size: 768
Model/tokenizer load time: 5.05s


## 6) Embedding Functions (Batch Encode + Pooling)


In [6]:
def pool_embeddings(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor, pooling: str = 'mean') -> torch.Tensor:
    if pooling == 'cls':
        return last_hidden_state[:, 0, :]
    if pooling == 'mean':
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts
    raise ValueError(f'Unsupported pooling: {pooling}')


def encode_texts(
    texts: list[str],
    tokenizer: AutoTokenizer,
    model: AutoModel,
    device: torch.device,
    batch_size: int = 32,
    max_length: int = 96,
    pooling: str = 'mean',
    use_fp16_if_cuda: bool = True,
) -> np.ndarray:
    start = time.perf_counter()
    outputs: list[np.ndarray] = []
    total = len(texts)
    n_batches = (total + batch_size - 1) // batch_size

    if total == 0:
        return np.empty((0, int(model.config.hidden_size)), dtype=np.float32)

    use_autocast = bool(device.type == 'cuda' and use_fp16_if_cuda)

    with torch.no_grad():
        for batch_idx in range(n_batches):
            b0 = batch_idx * batch_size
            b1 = min(total, (batch_idx + 1) * batch_size)
            batch_texts = texts[b0:b1]

            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}

            if use_autocast:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    model_out = model(**encoded)
            else:
                model_out = model(**encoded)

            pooled = pool_embeddings(
                last_hidden_state=model_out.last_hidden_state,
                attention_mask=encoded['attention_mask'],
                pooling=pooling,
            )
            outputs.append(pooled.detach().cpu().numpy().astype(np.float32))

            if batch_idx == 0 or (batch_idx + 1) % 20 == 0 or (batch_idx + 1) == n_batches:
                print(f'Batch {batch_idx + 1}/{n_batches} done (rows {b0}-{b1 - 1})')

    emb = np.concatenate(outputs, axis=0)
    elapsed = time.perf_counter() - start
    print(f'Embedding extraction completed in {elapsed:.2f}s')
    print(f'Embedding matrix shape: {emb.shape}')
    return emb


## 7) Extract BERT Embeddings


In [7]:
texts = df_text['text'].astype(str).tolist()

# Token-length diagnostics before extraction
length_probe = tokenizer(
    texts[: min(1000, len(texts))],
    padding=False,
    truncation=False,
    add_special_tokens=True,
)
probe_lengths = np.array([len(x) for x in length_probe['input_ids']], dtype=np.int32)

print(f'Rows to encode: {len(texts):,}')
if len(probe_lengths) > 0:
    print('Token-length probe on first sample rows:')
    print({
        'min': int(probe_lengths.min()),
        'p50': float(np.percentile(probe_lengths, 50)),
        'p90': float(np.percentile(probe_lengths, 90)),
        'p95': float(np.percentile(probe_lengths, 95)),
        'max': int(probe_lengths.max()),
        'max_length_setting': MAX_LENGTH,
    })

embeddings = encode_texts(
    texts=texts,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    pooling=POOLING,
    use_fp16_if_cuda=USE_FP16_IF_CUDA,
)

l2_norm = np.linalg.norm(embeddings, axis=1)
print('Embedding L2 norm summary:')
print(pd.Series(l2_norm).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


Rows to encode: 9,887
Token-length probe on first sample rows:
{'min': 3, 'p50': 13.0, 'p90': 35.0, 'p95': 43.0, 'max': 81, 'max_length_setting': 96}


Batch 1/155 done (rows 0-63)


Batch 20/155 done (rows 1216-1279)


Batch 40/155 done (rows 2496-2559)


Batch 60/155 done (rows 3776-3839)


Batch 80/155 done (rows 5056-5119)


Batch 100/155 done (rows 6336-6399)


Batch 120/155 done (rows 7616-7679)


Batch 140/155 done (rows 8896-8959)


Batch 155/155 done (rows 9856-9886)
Embedding extraction completed in 8.25s
Embedding matrix shape: (9887, 768)
Embedding L2 norm summary:
count    9887.000000
mean        9.477643
std         0.675323
min         7.609277
50%         9.417773
90%        10.365635
95%        10.687756
99%        11.414174
max        13.001986
dtype: float64


## 8) Assemble and Save Feature Artifacts


In [8]:
feature_cols = [f'bert_emb_{i:04d}' for i in range(embeddings.shape[1])]
emb_df = pd.DataFrame(embeddings, columns=feature_cols)

base_cols = ['path', 'session', 'method', 'gender', 'emotion', 'n_annotators', 'agreement', 'utt_id', 'text', 'split']
out_df = pd.concat([df_text[base_cols].reset_index(drop=True), emb_df], axis=1)

save_start = time.perf_counter()
out_df.to_csv(EMB_CSV, index=False)

metadata = {
    'model_name': MODEL_NAME,
    'pooling': POOLING,
    'max_length': int(MAX_LENGTH),
    'batch_size': int(BATCH_SIZE),
    'include_xxx': bool(INCLUDE_XXX),
    'require_agreement': bool(REQUIRE_AGREEMENT),
    'num_rows': int(len(out_df)),
    'embedding_dim': int(embeddings.shape[1]),
    'feature_columns': feature_cols,
}
META_JSON.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

elapsed = time.perf_counter() - save_start
print(f'Saved embedding CSV: {EMB_CSV}')
print(f'Saved metadata JSON: {META_JSON}')
print(f'Output shape: {out_df.shape}')
print(f'Save time: {elapsed:.2f}s')

out_df[['utt_id', 'emotion', 'split'] + feature_cols[:5]].head()


Saved embedding CSV: F:\Speech-Emotion-Recognition\extracted_features\text\bert_embeddings.csv
Saved metadata JSON: F:\Speech-Emotion-Recognition\extracted_features\text\bert_embeddings_metadata.json
Output shape: (9887, 778)
Save time: 5.70s


,utt_id,emotion,split,bert_emb_0000,bert_emb_0001,bert_emb_0002,bert_emb_0003,bert_emb_0004
0,Ses01F_script02_1_F000,neu,train,0.591567,-0.319067,0.056419,-0.127547,0.281918
1,Ses01F_script02_1_F001,fru,train,0.320125,0.217018,-0.023053,-0.354409,0.073762
2,Ses01F_script02_1_F002,xxx,train,0.310570,0.047458,-0.267729,-0.152551,-0.103211
3,Ses01F_script02_1_F004,neu,train,-0.080714,0.275665,-0.231021,-0.349853,0.301938
4,Ses01F_script02_1_F005,xxx,train,0.335354,0.397975,0.220571,-0.393551,0.005725
